In [27]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [28]:
openGeometry("boxes.geo")

In [29]:
#openPreProcessor()

In [30]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [31]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

0

In [32]:
C = contact(U, master="master", slave="slave")

Contact("slave" -> "master", 1957 candidate nodes, 0 active)

In [33]:
cn = 1e7
r = nodePositionVector(U)

nodal VectorField
[0.0; -2.0; … ; 1.795491126453324; 0.4787301686243378;;]

In [34]:
G = ContactGap(C)

Kc = ∫(G ⋅ cn ⋅ G)

rc = Kc * (r + u)

R = K * u - f + rc

nodal VectorField
[0.0; 0.0; … ; -1.0653370321625808e-12; 1.9642840855579234e-12;;]

In [35]:
support_increment = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uy=0, uz=0)
]


2-element Vector{BoundaryCondition}:
 BoundaryCondition("bottom", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0))
 BoundaryCondition("top", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0))

In [36]:
@time C = contact(
    u,
    master="master",
    slave="slave",
    topology_tol=0.01
)
u_it = copy(u)

@time G = ContactGap(C)

@time updateContact!(C, u_it)

@time Kc = ∫(G ⋅ cn ⋅ G)

rc = Kc * (r + u_it)
R  = K * u_it - f + rc

# a jelenlegi megoldási módoddal:
# Δu = ...
@time Δu = solveField(
        K + Kc,
        -R,
        support=support_increment
    )
    
u1 = u_it + Δu

@time updateContact!(C, u1)

@time gap1 = ContactGap(C, u1)

  0.378585 seconds (155.41 k allocations: 11.380 MiB)
  0.000012 seconds (9 allocations: 256 bytes)
  0.281045 seconds (157.04 k allocations: 11.464 MiB)
  1.380017 seconds (328.36 k allocations: 53.707 MiB, 5.95% gc time)
  5.366988 seconds (1.22 k allocations: 1.282 GiB, 4.80% gc time)
  0.281073 seconds (157.01 k allocations: 11.466 MiB)
  0.001250 seconds (24 allocations: 1.605 MiB)


nodal ScalarField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [37]:
Kc[:,:]

78834×78834 SparseMatrixCSC{Float64, Int64} with 19462 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⣤⣤⣤⠀⠀⣤⣤⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⣿⣿⣿⠀⠀⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⣿⣿⣿⠀⠀⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠛⠛⠛⠀⠀⠛⠛⠛⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [38]:
showElementResults(nodesToElements(gap1, onPhysicalGroup="slave"))

1

In [39]:
openPostProcessor()